# Gold — Métricas de negocio
**Proyecto:** AI Tech Landscape Pipeline  
**Descripción:** Calcula las métricas finales de negocio listas para visualización en Power BI.

In [ ]:
import pandas as pd
import os

SILVER_PATH = "../data/silver/"
GOLD_PATH   = "../data/gold/"
os.makedirs(GOLD_PATH, exist_ok=True)

df_stocks = pd.read_parquet(f"{SILVER_PATH}silver_big_tech_stocks.parquet")
df_gpu    = pd.read_parquet(f"{SILVER_PATH}silver_gpu_companies.parquet")
df_ai     = pd.read_parquet(f"{SILVER_PATH}silver_ai_companies.parquet")

print("✓ Datasets cargados")

## Big Tech — Precio promedio anual y variación %
Responde: ¿Cómo creció cada empresa año a año?

In [ ]:
df_stocks["year"] = df_stocks["date"].dt.year

# Precio promedio de cierre por empresa por año
gold_precio_anual = (
    df_stocks.groupby(["ticker", "year"])["close"]
    .mean()
    .reset_index()
    .rename(columns={"close": "avg_close_price"})
)

# Variación % vs año anterior
gold_precio_anual["pct_change_yoy"] = (
    gold_precio_anual
    .groupby("ticker")["avg_close_price"]
    .pct_change() * 100
).round(2)

gold_precio_anual["avg_close_price"] = gold_precio_anual["avg_close_price"].round(2)

print(f"Filas: {len(gold_precio_anual):,}")
gold_precio_anual.head(10)

## Big Tech — Mejor y peor año por empresa
Responde: ¿Cuándo tuvo cada empresa su pico y su caída más grande?

In [ ]:
mejor_anio = (
    gold_precio_anual.loc[gold_precio_anual.groupby("ticker")["pct_change_yoy"].idxmax()]
    [["ticker", "year", "pct_change_yoy"]]
    .rename(columns={"year": "mejor_anio", "pct_change_yoy": "mejor_retorno_pct"})
)

peor_anio = (
    gold_precio_anual.loc[gold_precio_anual.groupby("ticker")["pct_change_yoy"].idxmin()]
    [["ticker", "year", "pct_change_yoy"]]
    .rename(columns={"year": "peor_anio", "pct_change_yoy": "peor_retorno_pct"})
)

gold_performance = mejor_anio.merge(peor_anio, on="ticker")
print(f"Filas: {len(gold_performance)}")
gold_performance

## GPU Race — NVIDIA vs AMD vs INTEL evolución histórica
Responde: ¿Quién gana la carrera del hardware de IA?

In [ ]:
df_gpu["year"] = df_gpu["date"].dt.year

gold_gpu = (
    df_gpu.groupby(["empresa", "year"])["close"]
    .mean()
    .reset_index()
    .rename(columns={"close": "avg_close_price"})
)

gold_gpu["pct_change_yoy"] = (
    gold_gpu
    .groupby("empresa")["avg_close_price"]
    .pct_change() * 100
).round(2)

gold_gpu["avg_close_price"] = gold_gpu["avg_close_price"].round(2)

print(f"Empresas: {sorted(gold_gpu['empresa'].unique())}")
print(f"Filas   : {len(gold_gpu):,}")
gold_gpu.tail(10)

## AI Companies — Top 10 por revenue
Responde: ¿Quién factura más en el sector de IA?

In [ ]:
gold_top_revenue = (
    df_ai[["company_name", "revenue_usd", "founded", "headquarters"]]
    .dropna(subset=["revenue_usd"])
    .sort_values("revenue_usd", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

gold_top_revenue["revenue_usd_billions"] = (gold_top_revenue["revenue_usd"] / 1e9).round(2)

print(f"Top 10 empresas AI por revenue:")
gold_top_revenue[["company_name", "revenue_usd_billions", "founded", "headquarters"]]

## AI Companies — Revenue vs Glassdoor Score
Responde: ¿Las empresas que más facturan son las mejores para trabajar?

In [ ]:
gold_culture = (
    df_ai[["company_name", "revenue_usd", "glassdoor_score", "founded", "headquarters"]]
    .dropna(subset=["revenue_usd", "glassdoor_score"])
    .copy()
)

gold_culture["revenue_usd_billions"] = (gold_culture["revenue_usd"] / 1e9).round(2)
gold_culture["company_age"] = 2024 - gold_culture["founded"].astype(float)

print(f"Empresas con revenue y glassdoor: {len(gold_culture)}")
gold_culture[["company_name", "revenue_usd_billions", "glassdoor_score", "company_age"]].head(10)

## Guardar Gold

In [ ]:
gold_precio_anual.to_parquet(f"{GOLD_PATH}gold_bigtech_precio_anual.parquet",  index=False)
gold_performance.to_parquet(f"{GOLD_PATH}gold_bigtech_performance.parquet",    index=False)
gold_gpu.to_parquet(f"{GOLD_PATH}gold_gpu_race.parquet",                       index=False)
gold_top_revenue.to_parquet(f"{GOLD_PATH}gold_ai_top_revenue.parquet",         index=False)
gold_culture.to_parquet(f"{GOLD_PATH}gold_ai_culture_vs_money.parquet",        index=False)

print("=" * 50)
print("GOLD COMPLETADO")
print("=" * 50)
print(f"  Big Tech precio anual : {len(gold_precio_anual):,} filas")
print(f"  Big Tech performance  : {len(gold_performance):,} filas")
print(f"  GPU Race              : {len(gold_gpu):,} filas")
print(f"  AI Top Revenue        : {len(gold_top_revenue):,} filas")
print(f"  AI Culture vs Money   : {len(gold_culture):,} filas")
print("\n✓ Parquet listos para Power BI →")